In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from skopt import gp_minimize
from skopt.space import Real, Categorical
from skopt.utils import use_named_args
import warnings
import json
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei']  # Display Chinese text.
plt.rcParams['axes.unicode_minus'] = False  # Display minus signs.

In [ ]:
# ==================== 1. Read economic-cycle data ====================
sheet1_df = pd.read_excel('各类资产不同经济环境下表现.xlsx', sheet_name='Sheet1')
sheet2_df = pd.read_excel('各类资产不同经济环境下表现.xlsx', sheet_name='Sheet2')

# Remove rows containing the data-source marker.
sheet1_df = sheet1_df[~sheet1_df['指标名称'].astype(str).str.contains("数据来源", na=False)]
sheet2_df = sheet2_df[~sheet2_df['指标名称'].astype(str).str.contains("数据来源", na=False)]

# Convert dates.
sheet1_df['指标名称'] = pd.to_datetime(sheet1_df['指标名称'])
sheet2_df['指标名称'] = pd.to_datetime(sheet2_df['指标名称'])

In [ ]:
# ==================== 2. Read asset prices ====================
# Read the complete dataset.
asset_prices_df = pd.read_excel('asset_prices.xlsx').dropna().drop(columns=['中证REITs', 'WTI原油'])
asset_prices_df['日期'] = pd.to_datetime(asset_prices_df['日期'])
asset_prices_df.set_index('日期', inplace=True)
asset_prices_df = asset_prices_df.sort_index()

# Add momentum filters to the original prices.
# Momentum filter for the Nanhua Commodity Index
asset_prices_df['南华商品指数_动量过滤'] = np.where(
    asset_prices_df['南华商品指数'].rolling(1).mean().shift(1) > 0, 1, -1
)
asset_prices_df['南华商品指数_动量过滤'] = asset_prices_df['南华商品指数'] * asset_prices_df['南华商品指数_动量过滤'].fillna(1)

# Momentum filter for Shanghai gold
asset_prices_df['沪金_动量过滤'] = np.where(
    asset_prices_df['沪金'].rolling(2).mean().shift(1) > 0, 1, 0
)
asset_prices_df['沪金_动量过滤'] = asset_prices_df['沪金'] * asset_prices_df['沪金_动量过滤'].fillna(1)

# Full SVM training dataset, starting in 2008
monthly_prices = asset_prices_df.resample('ME').last()  # Month-end prices
monthly_returns = monthly_prices.pct_change().dropna()

# All-weather dataset from late 2009, aligned with macroeconomic data
aw_asset_prices = asset_prices_df.loc['2009-12-31':]
aw_monthly_prices = aw_asset_prices.resample('M').last()
aw_monthly_returns = aw_monthly_prices.pct_change().dropna()

aw_monthly_returns['南华商品指数_动量过滤'] = np.where(aw_monthly_returns['南华商品指数'].rolling(1).mean().shift(1)>0,1,-1)
aw_monthly_returns['南华商品指数_动量过滤']=aw_monthly_returns['南华商品指数']*aw_monthly_returns['南华商品指数_动量过滤'].fillna(1)

aw_monthly_returns['沪金_动量过滤'] = np.where(aw_monthly_returns['沪金'].rolling(2).mean().shift(1)>0,1,0)
aw_monthly_returns['沪金_动量过滤']=aw_monthly_returns['沪金']*aw_monthly_returns['沪金_动量过滤'].fillna(1)

In [ ]:
# ==================== 3. Merge macro states for the all-weather strategy ====================
sheet1_df.rename(columns={'指标名称': '日期', '高于预期为1，低于预期为2': '经济预期情况'}, inplace=True)
sheet2_df.rename(columns={'指标名称': '日期', '高于预期为1，低于预期为2': '通胀预期情况'}, inplace=True)

aw_monthly_returns = aw_monthly_returns.reset_index()
merged_df = aw_monthly_returns.merge(sheet1_df[['日期', '经济预期情况']], on='日期', how='left')
merged_df = merged_df.merge(sheet2_df[['日期', '通胀预期情况']], on='日期', how='left')

# Fill macroeconomic states.
merged_df['经济预期情况'] = merged_df['经济预期情况'].ffill()
merged_df['通胀预期情况'] = merged_df['通胀预期情况'].ffill()
merged_df[['经济预期情况', '通胀预期情况']] = merged_df[['经济预期情况', '通胀预期情况']].fillna(1)
merged_df = merged_df.dropna()

# Ensure the date column contains datetime values.
merged_df['日期'] = pd.to_datetime(merged_df['日期'])
merged_df.set_index('日期', inplace=True)

In [ ]:
# ==================== 5. Define the asset universe ====================
all_assets = [
    '中债国债', '中债金融债', '中债企业债', '南华商品指数_动量过滤',
    '中证转债', '红利收益', '上证50', '沪深300', '沪金_动量过滤', '中证1000'
]

In [ ]:
# ==================== 6. Technical-indicator functions ====================
def calculate_volatility(returns, window=12):
    """Calculate annualized volatility."""
    return returns.rolling(window).std() * np.sqrt(12)

def calculate_rsi(prices, window=6):
    """Calculate the relative strength index (RSI)."""
    delta = prices.diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    
    avg_gain = gain.rolling(window).mean()
    avg_loss = loss.rolling(window).mean()
    
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

In [ ]:
# ==================== 7. Construct SVM features and labels ====================
svm_features = {}
svm_labels = {}

for asset in all_assets:
    # Use the complete dataset starting in 2008.
    prices = monthly_prices[asset].dropna()
    rets = monthly_returns[asset].dropna()
    
    # Label: 1 if the next-period return is positive.
    y = (rets.shift(-1) > 0).astype(int)
    
    # Construct features.
    feat_df = pd.DataFrame(index=prices.index)
    
    # Basic features, lagged one period
    feat_df['ret_1m'] = rets.shift(1)                    # Previous month's return
    feat_df['ret_3m'] = rets.shift(1).rolling(3).mean()  # Mean return over the previous 3 months
    feat_df['ret_6m'] = rets.shift(1).rolling(6).mean()
    feat_df['ret_12m'] = rets.shift(1).rolling(12).mean()
    
    # Technical indicators, lagged one period
    feat_df['volatility_12m'] = calculate_volatility(rets, 12).shift(1)  # Annualized volatility
    feat_df['rsi_6m'] = calculate_rsi(prices, 6).shift(1)               # Six-month RSI
    
    # Drop initial NaN values.
    feat_df = feat_df.dropna()
    
    # Align labels.
    y = y.reindex(feat_df.index)
    
    svm_features[asset] = feat_df
    svm_labels[asset] = y

In [ ]:
# ==================== 8. Trading-oriented scoring function ====================
def custom_scorer(y_true, y_proba, threshold=0.5):
    y_pred = (y_proba >= threshold).astype(int)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_proba)
    
    down_mask = (y_true == 0)
    acc_down = (y_pred[down_mask] == 0).mean() if down_mask.sum() > 0 else 0
    
    score = 0.4 * acc + 0.3 * f1 + 0.3 * auc
    if acc_down < 0.3:
        score *= (acc_down / 0.3)
    return score

In [ ]:
# ==================== 9. Rolling training and prediction without future leakage ====================
# Initialize prediction storage.
svm_predictions = pd.DataFrame()
asset_pred_record = {asset: [] for asset in all_assets}
best_params_dict = {asset: None for asset in all_assets}

# Set rolling-training parameters.
INITIAL_TRAIN_START = '2008-01-01'  # Use data starting in 2008.
INITIAL_TRAIN_END = '2013-12-31'    # End of the initial training period
FIRST_PRED_DATE = '2014-01-01'      # First prediction date
RETRAIN_FREQ = 12                   # Re-optimize hyperparameters every 12 months.

# Train and predict separately for each asset.
for asset in all_assets:
    print(f"\n🔍 滚动训练预测中: {asset}")
    
    # Get features and labels for the asset.
    X = svm_features[asset]
    y = svm_labels[asset]
    
    # Align date indices.
    all_dates = X.index.sort_values()
    
    # Store predictions for this asset.
    asset_preds = pd.Series(index=all_dates, dtype=float)
    
    # Identify the first prediction date, January 2014.
    first_pred_date = pd.to_datetime(FIRST_PRED_DATE)
    
    # Initial training period, 2008-2013
    initial_train_dates = all_dates[(all_dates >= INITIAL_TRAIN_START) & (all_dates < first_pred_date)]
    X_initial = X.loc[initial_train_dates]
    y_initial = y.loc[initial_train_dates]
    
    # Drop NaN labels.
    valid_mask = y_initial.notna()
    X_initial = X_initial[valid_mask]
    y_initial = y_initial[valid_mask]
    
    # Skip the asset if the initial training sample is too small.
    if len(X_initial) < 30:
        print(f"⚠️  初始样本不足，跳过 {asset}")
        continue
    
    # Perform Bayesian optimization on the initial training period.
    print(f"  初始贝叶斯优化...")
    
    tscv = TimeSeriesSplit(n_splits=5)
    
    # Search space
    search_space = [
        Real(0.1, 100, prior='log-uniform', name='C'),
        Real(0.001, 1.0, prior='log-uniform', name='gamma'),
        Categorical(['None', 'balanced', '1:2', '1:3', '1:4'], name='class_weight_str')
    ]
    
    @use_named_args(search_space)
    def objective(**params):
        # Construct class weights.
        if params['class_weight_str'] == '1:2':
            class_weight = {0: 1, 1: 2}
        elif params['class_weight_str'] == '1:3':
            class_weight = {0: 1, 1: 3}
        elif params['class_weight_str'] == '1:4':
            class_weight = {0: 1, 1: 4}
        elif params['class_weight_str'] == 'balanced':
            class_weight = 'balanced'
        else:
            class_weight = None
            
        model = SVC(kernel='rbf', probability=True, C=params['C'], gamma=params['gamma'], class_weight=class_weight)
        calibrated_model = CalibratedClassifierCV(model, method='isotonic', cv=3)
        
        scores = []
        for train_idx, val_idx in tscv.split(X_initial):
            X_train, X_val = X_initial.iloc[train_idx], X_initial.iloc[val_idx]
            y_train, y_val = y_initial.iloc[train_idx], y_initial.iloc[val_idx]
            
            # Drop NaN labels.
            train_mask = y_train.notna()
            val_mask = y_val.notna()
            
            X_train = X_train[train_mask]
            y_train = y_train[train_mask]
            X_val = X_val[val_mask]
            y_val = y_val[val_mask]
            
            if len(X_train) == 0 or len(X_val) == 0:
                continue
                
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_val_scaled = scaler.transform(X_val)
            
            calibrated_model.fit(X_train_scaled, y_train)
            y_proba = calibrated_model.predict_proba(X_val_scaled)[:, 1]
            score = custom_scorer(y_val, y_proba)
            scores.append(score)
        
        return -np.mean(scores) if scores else 0
    
    # Run optimization.
    result = gp_minimize(
        func=objective,
        dimensions=search_space,
        n_calls=20,
        random_state=42,
        verbose=False
    )
    
    # Get the best parameters.
    best_params = dict(zip(['C', 'gamma', 'class_weight_str'], result.x))
    best_params_dict[asset] = best_params
    
    # Construct class weights.
    if best_params['class_weight_str'] == '1:2':
        class_weight = {0: 1, 1: 2}
    elif best_params['class_weight_str'] == '1:3':
        class_weight = {0: 1, 1: 3}
    elif best_params['class_weight_str'] == '1:4':
        class_weight = {0: 1, 1: 4}
    elif best_params['class_weight_str'] == 'balanced':
        class_weight = 'balanced'
    else:
        class_weight = None
        
    print(f"✅ 初始最优参数: C={best_params['C']:.3f}, gamma={best_params['gamma']:.3f}, class_weight={class_weight}")
    
    # Train with the initial parameters.
    model = SVC(kernel='rbf', probability=True, C=best_params['C'], gamma=best_params['gamma'], class_weight=class_weight)
    calibrated_model = CalibratedClassifierCV(model, method='isotonic', cv=5)
    
    scaler = StandardScaler()
    X_initial_scaled = scaler.fit_transform(X_initial)
    calibrated_model.fit(X_initial_scaled, y_initial)
    
    # Begin rolling prediction in January 2014.
    last_retrain_date = initial_train_dates[-1]
    
    for current_date in all_dates[all_dates >= first_pred_date]:
        # Check whether re-optimization is due.
        months_since_retrain = (current_date - last_retrain_date).days // 30
        retrain_needed = months_since_retrain >= RETRAIN_FREQ
        
        # Get training data through the previous month.
        train_dates = all_dates[all_dates < current_date]
        
        X_train = X.loc[train_dates]
        y_train = y.loc[train_dates]
        
        # Drop NaN labels.
        train_mask = y_train.notna()
        X_train = X_train[train_mask]
        y_train = y_train[train_mask]
        
        # Skip if there are too few observations.
        if len(X_train) < 30:
            continue
        
        # If re-optimization is required
        if retrain_needed:
            print(f"  🔄 重新优化超参数 (日期: {current_date.date()})")
            
            # Re-optimize on the current training set.
            @use_named_args(search_space)
            def current_objective(**params):
                # Construct class weights.
                if params['class_weight_str'] == '1:2':
                    class_weight = {0: 1, 1: 2}
                elif params['class_weight_str'] == '1:3':
                    class_weight = {0: 1, 1: 3}
                elif params['class_weight_str'] == '1:4':
                    class_weight = {0: 1, 1: 4}
                elif params['class_weight_str'] == 'balanced':
                    class_weight = 'balanced'
                else:
                    class_weight = None
                    
                model = SVC(kernel='rbf', probability=True, C=params['C'], gamma=params['gamma'], class_weight=class_weight)
                calibrated_model = CalibratedClassifierCV(model, method='isotonic', cv=3)
                
                scores = []
                for train_idx, val_idx in tscv.split(X_train):
                    X_train_fold, X_val_fold = X_train.iloc[train_idx], X_train.iloc[val_idx]
                    y_train_fold, y_val_fold = y_train.iloc[train_idx], y_train.iloc[val_idx]
                    
                    # Drop NaN labels.
                    train_mask = y_train_fold.notna()
                    val_mask = y_val_fold.notna()
                    
                    X_train_fold = X_train_fold[train_mask]
                    y_train_fold = y_train_fold[train_mask]
                    X_val_fold = X_val_fold[val_mask]
                    y_val_fold = y_val_fold[val_mask]
                    
                    if len(X_train_fold) == 0 or len(X_val_fold) == 0:
                        continue
                        
                    scaler_fold = StandardScaler()
                    X_train_scaled_fold = scaler_fold.fit_transform(X_train_fold)
                    X_val_scaled_fold = scaler_fold.transform(X_val_fold)
                    
                    calibrated_model.fit(X_train_scaled_fold, y_train_fold)
                    y_proba = calibrated_model.predict_proba(X_val_scaled_fold)[:, 1]
                    score = custom_scorer(y_val_fold, y_proba)
                    scores.append(score)
                
                return -np.mean(scores) if scores else 0
            
            # Run optimization.
            result = gp_minimize(
                func=current_objective,
                dimensions=search_space,
                n_calls=15,  # Reduce the number of calls for speed.
                random_state=42,
                verbose=False
            )
            
            # Update the best parameters.
            best_params = dict(zip(['C', 'gamma', 'class_weight_str'], result.x))
            best_params_dict[asset] = best_params
            
            # Construct class weights.
            if best_params['class_weight_str'] == '1:2':
                class_weight = {0: 1, 1: 2}
            elif best_params['class_weight_str'] == '1:3':
                class_weight = {0: 1, 1: 3}
            elif best_params['class_weight_str'] == '1:4':
                class_weight = {0: 1, 1: 4}
            elif best_params['class_weight_str'] == 'balanced':
                class_weight = 'balanced'
            else:
                class_weight = None
                
            print(f"  ✅ 新最优参数: C={best_params['C']:.3f}, gamma={best_params['gamma']:.3f}, class_weight={class_weight}")
            
            last_retrain_date = current_date
        
        # Train with the current best parameters.
        best_params = best_params_dict[asset]
        if best_params['class_weight_str'] == '1:2':
            class_weight = {0: 1, 1: 2}
        elif best_params['class_weight_str'] == '1:3':
            class_weight = {0: 1, 1: 3}
        elif best_params['class_weight_str'] == '1:4':
            class_weight = {0: 1, 1: 4}
        elif best_params['class_weight_str'] == 'balanced':
            class_weight = 'balanced'
        else:
            class_weight = None
            
        model = SVC(
            kernel='rbf', 
            probability=True, 
            C=best_params['C'], 
            gamma=best_params['gamma'], 
            class_weight=class_weight
        )
        calibrated_model = CalibratedClassifierCV(model, method='isotonic', cv=5)
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        calibrated_model.fit(X_train_scaled, y_train)
        
        # Predict at the current date, targeting the next month.
        X_current = X.loc[[current_date]].values.reshape(1, -1)
        X_current_scaled = scaler.transform(X_current)
        pred_proba = calibrated_model.predict_proba(X_current_scaled)[0, 1]
        
        # Store the prediction.
        asset_preds[current_date] = pred_proba
        svm_predictions.loc[current_date, asset] = pred_proba
        
        # Record the prediction.
        true_label = y.loc[current_date] if current_date in y.index else np.nan
        pred_label = 1 if pred_proba >= 0.5 else 0
        
        asset_pred_record[asset].append({
            'date': current_date,
            'true': true_label,
            'pred': pred_label,
            'proba': pred_proba
        })
    
    # Store predictions for this asset.
    svm_predictions[asset] = asset_preds

# Save the best parameters.
with open('best_svm_params.json', 'w') as f:
    json.dump(best_params_dict, f, indent=4)


In [ ]:
# ==================== 10. Construct synthetic quadrant assets ====================
asset_grid = {
    (1, 1): ['南华商品指数_动量过滤', '沪金_动量过滤', '上证50', '红利收益', '中证转债'],
    (1, 2): ['沪深300', '上证50', '中证1000', '中证转债', '中债企业债'],
    (2, 1): ['南华商品指数_动量过滤', '沪金_动量过滤', '中债金融债'],
    (2, 2): ['中债国债', '中债金融债', '中债企业债', '红利收益']
}

LOOKBACK_PERIOD = 12
quad_cols = []

for key, assets in asset_grid.items():
    vol = merged_df[assets].rolling(window=LOOKBACK_PERIOD).std() * np.sqrt(12)
    vol = vol.shift(1)
    inv_vol = 1 / vol
    weight = inv_vol.div(inv_vol.sum(axis=1), axis=0)
    merged_df[f'新资产_{key}'] = (merged_df[assets] * weight).sum(axis=1)
    quad_cols.append(f'新资产_{key}')

In [ ]:
# ==================== 11. All-weather portfolio with risk parity across quadrants ====================
quad_vol = merged_df[quad_cols].rolling(window=LOOKBACK_PERIOD).std() * np.sqrt(12)
quad_vol = quad_vol.shift(1)
top_inv_vol = 1 / quad_vol
top_weights = top_inv_vol.div(top_inv_vol.sum(axis=1), axis=0)
merged_df['全天候组合收益'] = (merged_df[quad_cols] * top_weights).sum(axis=1)

In [ ]:
def perf_stats(nav: pd.Series, freq: int = 12) -> pd.Series:
    """
    Basic performance metrics:
    Annualized return, annualized volatility, Sharpe ratio with a zero risk-free rate,
    and maximum drawdown.
    """
    rtn = nav.pct_change().dropna()
    ann_ret = nav.iloc[-1]**(freq/len(rtn)) - 1
    ann_vol = rtn.std() * np.sqrt(freq)
    sharpe  = ann_ret / ann_vol if ann_vol != 0 else np.nan
    drawdown = nav / nav.cummax() - 1
    max_dd = drawdown.min()
    stats = pd.Series({"累积收益": nav.iloc[-1] - 1,
                       "年化收益": ann_ret,
                       "年化波动": ann_vol,
                       "夏普比率": sharpe,
                       "最大回撤": max_dd})
    return stats
perf_stats((1 + merged_df['全天候组合收益'][48:]).cumprod(),12)

In [ ]:
# ==================== 12. SVM-filtered portfolio ====================
svm_weights = pd.DataFrame(0.0, index=merged_df.index, columns=all_assets)

# Precompute volatility with a minimum floor.
MIN_VOLATILITY = 0.001  # Minimum volatility prevents division by zero.
asset_vol = merged_df[all_assets].rolling(window=LOOKBACK_PERIOD).std() * np.sqrt(12)
asset_vol = asset_vol.shift(1)
asset_vol = asset_vol.fillna(asset_vol.mean())  # Fill NaN values with the mean.
asset_vol = asset_vol.clip(lower=MIN_VOLATILITY)  # Apply the volatility floor.

# Use dates from 2014 onward.
start_date = pd.Timestamp('2014-01-01')
prediction_dates = merged_df.index[merged_df.index >= start_date]

# Use only dates present in both datasets.
common_dates = prediction_dates.intersection(svm_predictions.index)

for idx in common_dates:
    # Get predictions for this date.
    pred_row = svm_predictions.loc[idx]
    
    # Select assets whose predicted probability exceeds 0.5.
    selected_assets = [
        asset for asset in all_assets
        if pd.notna(pred_row[asset]) and pred_row[asset] > 0.5
    ]
    
    if not selected_assets:
        continue
    
    # Get volatility.
    vol_values = asset_vol.loc[idx, selected_assets]
    
    # Calculate inverse-volatility weights.
    inv_vol = 1 / vol_values
    total_inv_vol = inv_vol.sum()
    
    if total_inv_vol > 0:
        weights = inv_vol / total_inv_vol
    else:
        # Use equal weights if the inverse-volatility sum is zero.
        weights = pd.Series(1/len(selected_assets), index=selected_assets)
    
    svm_weights.loc[idx, selected_assets] = weights.values

# SVM portfolio return
svm_portfolio_returns = pd.Series(0.0, index=merged_df.index)
for idx in common_dates:
    weights = svm_weights.loc[idx]
    returns = merged_df.loc[idx, all_assets]
    svm_portfolio_returns.loc[idx] = (weights * returns).sum()

merged_df['SVM筛选组合收益'] = svm_portfolio_returns

In [ ]:
# ==================== 13. Backtest analysis ====================
# Analyze dates from 2014 onward only.
backtest_df = merged_df[merged_df.index >= '2014-01-01']

# Cumulative returns
backtest_df['SVM筛选累计收益'] = (1 + backtest_df['SVM筛选组合收益']).cumprod()
backtest_df['全天候累计收益'] = (1 + backtest_df['全天候组合收益']).cumprod()

# Plot results.
plt.figure(figsize=(12, 6))
plt.plot(backtest_df.index, backtest_df['全天候累计收益'], label='全天候策略', linewidth=2, color='blue')
plt.plot(backtest_df.index, backtest_df['SVM筛选累计收益'], label='SVM筛选策略', linewidth=2, color='red')
plt.xlabel('日期')
plt.ylabel('累计收益')
plt.title('全天候策略 vs SVM筛选策略 (2014-至今)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
#plt.savefig('strategy_comparison.png', dpi=300)
plt.show()

In [ ]:
# Performance metrics
def calculate_performance(returns, freq=12):
    cumulative_return = (1 + returns).prod() - 1
    annual_return = (1 + cumulative_return) ** (freq / len(returns)) - 1
    annual_volatility = returns.std() * np.sqrt(freq)
    sharpe_ratio = annual_return / annual_volatility if annual_volatility != 0 else 0
    wealth_index = (1 + returns).cumprod()
    previous_peaks = wealth_index.cummax()
    drawdowns = (wealth_index - previous_peaks) / previous_peaks
    max_drawdown = drawdowns.min()
    return {
        '累计收益': cumulative_return,
        '年化收益': annual_return,
        '年化波动': annual_volatility,
        '夏普比率': sharpe_ratio,
        '最大回撤': max_drawdown
    }

aw_perf = calculate_performance(backtest_df['全天候组合收益'].dropna())
svm_perf = calculate_performance(backtest_df['SVM筛选组合收益'].dropna())

perf_df = pd.DataFrame([aw_perf, svm_perf], index=['全天候策略', 'SVM筛选策略'])
print(perf_df)

In [ ]:
# ==================== 14. SVM prediction performance by asset ====================
performance_summary = []
rolling_acc_dict = {}  # Store rolling accuracy for each asset.
prediction_records = []  # Store prediction records for later analysis.

for asset in all_assets:
    records = asset_pred_record.get(asset, [])
    if not records:
        continue
        
    dates = [r['date'] for r in records]
    y_true = np.array([r['true'] for r in records])
    y_pred = np.array([r['pred'] for r in records])
    y_proba = np.array([r['proba'] for r in records])
    
    # Calculate basic metrics.
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_proba)
    hit_rate = (y_proba > 0.5).mean()  # Fraction with predicted up probability above 0.5
    
    # Calculate rolling accuracy over 12 months.
    rolling_acc = pd.Series([int(p == t) for p, t in zip(y_pred, y_true)], index=dates)
    rolling_acc = rolling_acc.rolling(window=12, min_periods=6).mean()
    rolling_acc_dict[asset] = rolling_acc
    
    # Store records for later analysis.
    for record in records:
        prediction_records.append({
            'asset': asset,
            'date': record['date'],
            'true': record['true'],
            'pred': record['pred'],
            'proba': record['proba'],
            'correct': int(record['pred'] == record['true'])
        })
    
    performance_summary.append({
        '资产': asset,
        '样本数': len(y_true),
        '准确率': acc,
        'F1分数': f1,
        'AUC': auc,
        '上涨预测占比': hit_rate
    })

# Convert to a DataFrame for analysis.
perf_df_by_asset = pd.DataFrame(performance_summary).sort_values('AUC', ascending=False)

In [ ]:
# ==================== 15. Statistical tests ====================
# Test whether strategy returns differ significantly.
t_stat, p_value = ttest_ind(
    backtest_df['SVM筛选组合收益'].dropna(), 
    backtest_df['全天候组合收益'].dropna(),
    equal_var=False
)

In [ ]:
# ==================== 16. Compact output ====================
print("\n" + "="*50)
print("📊 策略绩效对比 (2015-至今):")
print("="*50)
print(perf_df.round(4))
print("\n" + "="*50)

print("\n📈 关键策略表现指标:")
print("-"*50)
print(f"全天候策略: 年化收益 = {perf_df.loc['全天候策略', '年化收益']:.4f}, 夏普比率 = {perf_df.loc['全天候策略', '夏普比率']:.4f}")
print(f"SVM筛选策略: 年化收益 = {perf_df.loc['SVM筛选策略', '年化收益']:.4f}, 夏普比率 = {perf_df.loc['SVM筛选策略', '夏普比率']:.4f}")
print("-"*50)

print("\n📊 策略收益差异显著性检验:")
print(f"SVM筛选 vs 全天候: t-statistic={t_stat:.3f}, p-value={p_value:.3f}")
print("="*50)

print("\n📈 各资产 SVM 预测性能 (按 AUC 排序):")
print("="*50)
print(perf_df_by_asset[['资产', 'AUC', '准确率', 'F1分数', '上涨预测占比']].round(3).to_string(index=False))
print("="*50)

In [ ]:
print(perf_df_by_asset[['资产', 'AUC', '准确率', 'F1分数', '上涨预测占比']].round(3).to_string(index=False))

In [ ]:
# ==================== 18. Compare allocation weights with consistent stacked-area colors ====================
print("\n" + "="*50)
print("📊 资产配置权重对比分析 - 堆叠面积图（统一颜色）")
print("="*50)

# Prepare the overall all-weather weight table.
all_assets = sorted({a for lst in asset_grid.values() for a in lst})
total_weights_aw = pd.DataFrame(0.0, index=merged_df.index, columns=all_assets)

within_weights = {}           # Store within-quadrant weights as DataFrames.
quad_col_map = {}             # Map each key to its synthetic quadrant-asset column name.
new_asset_cols = []           # Names of all representative quadrant-asset columns

for key, assets in asset_grid.items():
    # 1) Within quadrants: rolling inverse-volatility weights
    vol = merged_df[assets].rolling(window=LOOKBACK_PERIOD).std() * np.sqrt(12)
    vol = vol.shift(1)  # Use previous-period volatility to calculate weights.
    inv_vol_weight = 1 / vol
    weight_in = inv_vol_weight.div(inv_vol_weight.sum(axis=1), axis=0)
    weight_in = weight_in.fillna(0)  # Fill NaN values with zero.
    
    within_weights[key] = weight_in

    # 2) Create the synthetic quadrant-return column if it is absent.
    quad_col = f'新资产_{key}'
    quad_col_map[key] = quad_col
    new_asset_cols.append(quad_col)

# 3) Apply inverse-volatility weights across the four synthetic quadrant assets.
quad_vol = merged_df[new_asset_cols].rolling(window=LOOKBACK_PERIOD).std() * np.sqrt(12)
quad_vol = quad_vol.shift(1)  # Use previous-period volatility to calculate weights.
quad_inv = 1 / quad_vol
top_weights = quad_inv.div(quad_inv.sum(axis=1), axis=0)  # Normalize each row to sum to one.
top_weights = top_weights.fillna(0)  # Fill NaN values with zero.

# 4) Component weight = within-quadrant weight times top-level quadrant weight.
for key, assets in asset_grid.items():
    top_w_series = top_weights[quad_col_map[key]]  
    adj = within_weights[key].mul(top_w_series, axis=0) 
    
    # Add component weights for this quadrant to the overall weight table.
    for asset in assets:
        if asset in total_weights_aw.columns:
            total_weights_aw[asset] = total_weights_aw[asset].add(adj[asset], fill_value=0)

# 5) Prepare the overall SVM strategy weight table.
total_weights_svm = svm_weights.copy()

# Align both weight-table time indices from 2015 onward.
start_date = pd.Timestamp('2014-01-01')
total_weights_aw = total_weights_aw.loc[start_date:]
total_weights_svm = total_weights_svm.loc[start_date:]

# 6) Assign a fixed color to each asset.
# Use tab20 to give each asset a unique, consistent color.
asset_colors = plt.cm.tab20(np.linspace(0, 1, len(all_assets)))
color_map = {asset: color for asset, color in zip(all_assets, asset_colors)}

# 7) Compare stacked-area plots.
plt.figure(figsize=(16, 12))

# Panel 1: all-weather asset weights.
plt.subplot(2, 1, 1)
# Order assets according to the color mapping.
colors_aw = [color_map[asset] for asset in total_weights_aw.columns]
plt.stackplot(total_weights_aw.index, total_weights_aw.values.T, 
              labels=total_weights_aw.columns, colors=colors_aw)
plt.title('全天候策略资产权重分布', fontsize=14)
plt.xlabel('日期')
plt.ylabel('资产权重')
plt.legend(loc='upper left', bbox_to_anchor=(1.05, 1))
plt.grid(alpha=0.3)
plt.xlim(total_weights_aw.index[0], total_weights_aw.index[-1])

# Panel 2: SVM-filtered asset weights.
plt.subplot(2, 1, 2)
# Use the same color order.
# Select assets actually present in the SVM strategy.
svm_assets = [asset for asset in all_assets if asset in total_weights_svm.columns]
colors_svm = [color_map[asset] for asset in svm_assets]
plt.stackplot(total_weights_svm.index, total_weights_svm[svm_assets].values.T, 
              labels=svm_assets, colors=colors_svm)
plt.title('SVM筛选策略资产权重分布', fontsize=14)
plt.xlabel('日期')
plt.ylabel('资产权重')
plt.legend(loc='upper left', bbox_to_anchor=(1.05, 1))
plt.grid(alpha=0.3)
plt.xlim(total_weights_svm.index[0], total_weights_svm.index[-1])

plt.tight_layout()
plt.savefig('策略权重对比_堆叠面积图_统一颜色.png', dpi=300, bbox_inches='tight')
plt.show()



# Plot asset-weight heatmaps.
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 12))

# SVM strategy weight heatmap
im1 = ax1.imshow(svm_weights.T, aspect='auto', cmap='YlOrRd')
ax1.set_yticks(range(len(all_assets)))
ax1.set_yticklabels(all_assets)
ax1.set_title('SVM策略资产权重热力图')
fig.colorbar(im1, ax=ax1)

# All-weather strategy weight heatmap
im2 = ax2.imshow(aw_asset_weights.T, aspect='auto', cmap='YlOrRd')
ax2.set_yticks(range(len(all_assets)))
ax2.set_yticklabels(all_assets)
ax2.set_title('全天候策略资产权重热力图')
fig.colorbar(im2, ax=ax2)

plt.tight_layout()
#plt.savefig('weight_heatmap.png', dpi=300)
plt.show()

# Calculate weight-stability metrics.
def calculate_weight_stability(weights):
    """Calculate weight-stability metrics."""
    weight_changes = weights.diff().abs().mean(axis=1)
    turnover = weight_changes.mean()
    return turnover

svm_turnover = calculate_weight_stability(svm_weights)
aw_turnover = calculate_weight_stability(aw_asset_weights)

# Calculate each asset's selection frequency.
svm_selection_freq = (svm_weights > 0).mean()
aw_selection_freq = (aw_asset_weights > 0).mean()

selection_df = pd.DataFrame({
    'SVM选择频率': svm_selection_freq,
    '全天候选择频率': aw_selection_freq,
    '选择频率差异': svm_selection_freq - aw_selection_freq
}).sort_values('选择频率差异', ascending=False)

# Print weight-analysis results.
print("\n📈 权重稳定性分析:")
print("-"*50)
print(f"SVM策略换手率: {svm_turnover:.4f}")
print(f"全天候策略换手率: {aw_turnover:.4f}")
print(f"换手率差异: {svm_turnover - aw_turnover:.4f}")
print("-"*50)

print("\n📊 资产选择频率对比 (SVM vs 全天候):")
print("-"*50)
print(selection_df.round(3))
print("-"*50)
'''
# Save weight data.
try:
    svm_weights.to_csv('svm_weights.csv')
    aw_asset_weights.to_csv('all_weather_weights.csv')
    weight_comparison_df.to_csv('weight_comparison.csv')
    selection_df.to_csv('asset_selection_frequency.csv')
    print("Weight-analysis results saved to CSV.")
except PermissionError:
    print("Could not save weight analysis; another program may have the file open.")

print("Asset-allocation weight comparison complete.")
'''

In [ ]:
# ==================== 19. Weight-adjusted cumulative NAV contributions by asset ====================
print("\n" + "="*50)
print("📊 策略中各资产的累计净值贡献（权重调整后）")
print("="*50)

# Define the assets to display.
assets_list = [
    '上证50', '沪深300', '中证1000', '红利收益', '南华商品指数_动量过滤', 
    '中证转债', '中债国债', '中债企业债', '中债金融债', '沪金_动量过滤'
]

# Use the same color mapping.
if 'color_map' not in locals():
    # Create a color mapping if none exists.
    asset_colors = plt.cm.tab20(np.linspace(0, 1, len(assets_list)))
    color_map = {asset: color for asset, color in zip(assets_list, asset_colors)}

# Prepare data from 2015 onward.
start_date = pd.Timestamp('2014-01-01')
asset_returns = merged_df.loc[start_date:, assets_list]

# Calculate asset-level cumulative NAV in the SVM strategy.
svm_cumulative = pd.DataFrame(index=asset_returns.index)
for asset in assets_list:
    # Calculate this asset's NAV contribution to the SVM strategy.
    weighted_returns = asset_returns[asset] * svm_weights[asset]
    svm_cumulative[asset] = (1 + weighted_returns).cumprod()

# Calculate asset-level cumulative NAV in the all-weather strategy.
aw_cumulative = pd.DataFrame(index=asset_returns.index)
for asset in assets_list:
    # Calculate this asset's NAV contribution to the all-weather strategy.
    weighted_returns = asset_returns[asset] * total_weights_aw[asset]
    aw_cumulative[asset] = (1 + weighted_returns).cumprod()

# Plot asset-level cumulative NAV contributions for the SVM strategy.
plt.figure(figsize=(14, 7))
for asset in assets_list:
    plt.plot(svm_cumulative.index, svm_cumulative[asset], 
             label=asset, color=color_map[asset], linewidth=2)

# Add total SVM strategy NAV as a reference.
# Plot total cumulative SVM strategy returns as an optional reference.
         # Optional reference styling: black dashed line, linewidth=2.5.

# Set chart properties.
plt.title('SVM策略中各资产的累计净值贡献 (2014-至今)', fontsize=16)
plt.xlabel('日期', fontsize=12)
plt.ylabel('累计净值', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(loc='upper left', bbox_to_anchor=(1.05, 1))
plt.tight_layout()

# Save the chart.
#plt.savefig('svm_asset_nav_contributions.png', dpi=300, bbox_inches='tight')
plt.show()

# Plot asset-level cumulative NAV contributions for the all-weather strategy.
plt.figure(figsize=(14, 7))
for asset in assets_list:
    plt.plot(aw_cumulative.index, aw_cumulative[asset], 
             label=asset, color=color_map[asset], linewidth=2)

# Add total all-weather NAV as a reference.
# Plot total cumulative all-weather returns as an optional reference.
         # Optional reference styling: black dashed line, linewidth=2.5.

# Set chart properties.
plt.title('全天候策略中各资产的累计净值贡献 (2014-至今)', fontsize=16)
plt.xlabel('日期', fontsize=12)
plt.ylabel('累计净值', fontsize=12)
plt.grid(True, alpha=0.3)
plt.legend(loc='upper left', bbox_to_anchor=(1.05, 1))
plt.tight_layout()

# Save the chart.
#plt.savefig('all_weather_asset_nav_contributions.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ 策略中各资产的累计净值贡献图已保存")

In [ ]:
# Create the prediction-record DataFrame.
pred_df = pd.DataFrame(prediction_records)
pred_df['date'] = pd.to_datetime(pred_df['date'])
pred_df.set_index('date', inplace=True)

# 1. Overall prediction accuracy over time
plt.figure(figsize=(14, 6))
overall_acc = pred_df.groupby('date')['correct'].mean().rolling(window=12, min_periods=6).mean()
overall_acc.plot(title='整体预测准确率（12个月滚动平均）', color='blue', linewidth=2)
plt.xlabel('日期')
plt.ylabel('准确率')
plt.grid(True, alpha=0.3)
plt.axhline(y=0.5, color='red', linestyle='--', alpha=0.5)
plt.tight_layout()
#plt.savefig('overall_prediction_accuracy.png', dpi=300)
plt.show()

# 2. Prediction accuracy by asset over time
plt.figure(figsize=(14, 10))
for i, asset in enumerate(all_assets):
    if asset in rolling_acc_dict:
        plt.subplot(5, 2, i+1)
        rolling_acc_dict[asset].plot(title=f'{asset}预测准确率', color='green', linewidth=1.5)
        plt.axhline(y=0.5, color='red', linestyle='--', alpha=0.5)
        plt.grid(True, alpha=0.3)
        plt.ylim(0.3, 0.9)
plt.tight_layout()
#plt.savefig('prediction_accuracy_by_asset.png', dpi=300)
plt.show()

# 3. Predicted probabilities versus actual labels
plt.figure(figsize=(14, 8))
for i, asset in enumerate(all_assets[:4]):  # Show only the first four assets.
    if asset in rolling_acc_dict:
        asset_df = pred_df[pred_df['asset'] == asset]
        plt.subplot(2, 2, i+1)
        plt.scatter(asset_df.index, asset_df['proba'], 
                    c=asset_df['true'], cmap='coolwarm', alpha=0.6)
        plt.title(f'{asset}预测概率 vs 真实标签')
        plt.xlabel('日期')
        plt.ylabel('预测概率')
        plt.axhline(y=0.5, color='black', linestyle='--', alpha=0.5)
        plt.grid(True, alpha=0.3)
        plt.colorbar(label='真实标签 (0=下跌, 1=上涨)')
plt.tight_layout()
#plt.savefig('predictions_vs_labels.png', dpi=300)
plt.show()

# 4. Analyze the effect of retraining.
retrain_dates = []
for asset in all_assets:
    records = asset_pred_record.get(asset, [])
    if not records:
        continue
    
    # Extract retraining dates.
    retrain_dates_asset = []
    last_train_date = None
    for record in records:
        current_date = record['date']
        if last_train_date:
            months_since = (current_date - last_train_date).days // 30
            if months_since >= RETRAIN_FREQ:
                retrain_dates_asset.append(current_date)
                last_train_date = current_date
        else:
            last_train_date = current_date
    
    # Compare accuracy before and after retraining.
    if retrain_dates_asset:
        retrain_perf = []
        for retrain_date in retrain_dates_asset:
            # Six months before retraining
            pre_period = pred_df[(pred_df.index < retrain_date) & 
                                (pred_df.index >= retrain_date - pd.DateOffset(months=6)) &
                                (pred_df['asset'] == asset)]
            pre_acc = pre_period['correct'].mean() if not pre_period.empty else np.nan
            
            # Six months after retraining
            post_period = pred_df[(pred_df.index > retrain_date) & 
                                 (pred_df.index <= retrain_date + pd.DateOffset(months=6)) &
                                 (pred_df['asset'] == asset)]
            post_acc = post_period['correct'].mean() if not post_period.empty else np.nan
            
            if not np.isnan(pre_acc) and not np.isnan(post_acc):
                retrain_perf.append({
                    'date': retrain_date,
                    'pre_acc': pre_acc,
                    'post_acc': post_acc,
                    'delta': post_acc - pre_acc
                })
        
        if retrain_perf:
            retrain_df = pd.DataFrame(retrain_perf)
            plt.figure(figsize=(10, 6))
            plt.bar(retrain_df['date'], retrain_df['delta'], 
                   color=np.where(retrain_df['delta'] > 0, 'green', 'red'))
            plt.title(f'{asset}重新训练对准确率的影响')
            plt.xlabel('重新训练日期')
            plt.ylabel('准确率变化 (后6月 - 前6月)')
            plt.axhline(y=0, color='black', linestyle='-', alpha=0.5)
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            #plt.savefig(f'{asset}_retraining_effect.png', dpi=300)
            plt.show()
            
            # Calculate the mean change.
            avg_delta = retrain_df['delta'].mean()
            print(f"{asset}: 重新训练后准确率平均变化: {avg_delta:.4f}")

# ==================== 19. Prediction-performance heatmap ====================
# Create the prediction-performance matrix.
pred_matrix = pred_df.pivot_table(index='date', columns='asset', values='correct')
pred_matrix = pred_matrix.rolling(window=6, min_periods=3).mean()  # Six-month rolling mean

plt.figure(figsize=(14, 8))
plt.imshow(pred_matrix.T, aspect='auto', cmap='RdYlGn', vmin=0.4, vmax=0.8)
plt.colorbar(label='准确率')
plt.title('各资产预测准确率热力图 (6个月滚动平均)')
plt.xlabel('日期')
plt.ylabel('资产')
plt.xticks(range(0, len(pred_matrix), 12), pred_matrix.index[::12].strftime('%Y-%m'), rotation=45)
plt.yticks(range(len(pred_matrix.columns)), pred_matrix.columns)
plt.tight_layout()
#plt.savefig('prediction_accuracy_heatmap.png', dpi=300)
plt.show()

# ==================== 20. Prediction-confidence analysis ====================
plt.figure(figsize=(14, 8))
for i, asset in enumerate(all_assets[:4]):  # Show only the first four assets.
    if asset in rolling_acc_dict:
        asset_df = pred_df[pred_df['asset'] == asset]
        plt.subplot(2, 2, i+1)
        
        # Group accuracy by predicted probability.
        prob_bins = np.arange(0, 1.1, 0.1)
        bin_centers = (prob_bins[:-1] + prob_bins[1:]) / 2
        acc_by_bin = []
        
        for j in range(len(prob_bins)-1):
            low = prob_bins[j]
            high = prob_bins[j+1]
            bin_data = asset_df[(asset_df['proba'] >= low) & (asset_df['proba'] < high)]
            if not bin_data.empty:
                acc = bin_data['correct'].mean()
                acc_by_bin.append(acc)
            else:
                acc_by_bin.append(np.nan)
        
        plt.plot(bin_centers, acc_by_bin, 'o-', label='实际准确率')
        plt.plot([0, 1], [0, 1], 'k--', label='理想情况')
        plt.title(f'{asset}预测置信度校准')
        plt.xlabel('预测概率')
        plt.ylabel('实际准确率')
        plt.grid(True, alpha=0.3)
        plt.legend()
plt.tight_layout()
#plt.savefig('prediction_confidence_calibration.png', dpi=300)
plt.show()